# Impact of Gate Ordering on Quantum Circuit Compilation

This experiment studies whether changing the order of commuting quantum gates can affect the final compiled circuit without changing the logical behavior of the circuit.

**Research question:** Can different orderings of equivalent CZ gates produce different physical circuits after compilation?

Metrics: routing SWAP count, routing-stage depth, final native-circuit depth, and final CX count.


In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap

colors = {
    "original": "#7A7A7A",
    "random": "#4C78A8",
    "heuristic": "#F58518"
}

legend_elements = [
    Patch(facecolor=colors["original"], label="Initial ordering"),
    Patch(facecolor=colors["random"], label="Random orderings"),
    Patch(facecolor=colors["heuristic"], label="Heuristic ordering"),
    Line2D([0], [0], color="#E45756", linestyle="--", linewidth=2, label="Random average")
]


## Logical circuit and candidate orderings

The experiment uses a 6-qubit circuit containing 10 CZ gates. Since CZ gates commute, reordering this gate set preserves the logical operation.


In [ ]:
original_circuit = QuantumCircuit(6)

cz_gates = [(0, 3), (1, 3), (1, 2), (2, 4), (5, 2),
            (2, 0), (4, 0), (3, 5), (4, 5), (1, 4)]

for q1, q2 in cz_gates:
    original_circuit.cz(q1, q2)

original_circuit.draw("mpl")


In [ ]:
random.seed(42)
random_orderings = []

for _ in range(20):
    randgates = cz_gates.copy()
    random.shuffle(randgates)
    random_orderings.append(randgates)

heuristic_order = []
remaining_gates = cz_gates.copy()
current_gate = remaining_gates[0]
heuristic_order.append(current_gate)
remaining_gates.remove(current_gate)

while remaining_gates:
    best_candidate = None
    best_score = None

    for candidate in remaining_gates:
        for i in range(2):
            for j in range(2):
                if current_gate[i] == candidate[j]:
                    old_partner = current_gate[1 - i]
                    new_partner = candidate[1 - j]
                    distance = abs(old_partner - new_partner)
                    candidate_distance = abs(candidate[0] - candidate[1])
                    score = (distance, candidate_distance)

                    if best_score is None or score < best_score:
                        best_candidate = candidate
                        best_score = score

    if best_candidate is None:
        best_candidate = remaining_gates[0]

    heuristic_order.append(best_candidate)
    remaining_gates.remove(best_candidate)
    current_gate = best_candidate

orderings = [cz_gates, *random_orderings, heuristic_order]
len(orderings)


## Routing-stage experiment

The hardware topology, initial layout, optimization level, routing method, and transpiler-seed set are fixed. Only gate ordering changes.


In [ ]:
coupling_map = CouplingMap.from_line(6)
results_all = []

for seed in [0, 1, 2, 3, 4]:
    for i in range(22):
        circuit = QuantumCircuit(6)
        for q1, q2 in orderings[i]:
            circuit.cz(q1, q2)

        physical = transpile(
            circuit,
            coupling_map=coupling_map,
            optimization_level=1,
            initial_layout=[0, 1, 2, 3, 4, 5],
            routing_method="sabre",
            seed_transpiler=seed
        )

        ordering_type = "original" if i == 0 else ("heuristic" if i == 21 else "random")
        results_all.append({
            "ordering_id": i,
            "depth": physical.depth(),
            "swap_count": physical.count_ops().get("swap", 0),
            "size": physical.size(),
            "ordering_type": ordering_type,
            "seed": seed
        })

df = pd.DataFrame(results_all)

summary = (
    df.groupby(["ordering_id", "ordering_type"])
      .agg(
          mean_depth=("depth", "mean"),
          mean_swap=("swap_count", "mean"),
          std_depth=("depth", "std"),
          std_swap=("swap_count", "std")
      )
      .reset_index()
)

type_summary = (
    df.groupby("ordering_type")
      .agg(
          mean_depth=("depth", "mean"),
          mean_swap=("swap_count", "mean"),
          std_depth=("depth", "std"),
          std_swap=("swap_count", "std")
      )
      .reset_index()
)

type_summary


In [ ]:
routing_plot = summary.sort_values("ordering_id")
bar_colors = [colors[t] for t in routing_plot["ordering_type"]]
random_avg_depth = routing_plot[routing_plot["ordering_type"] == "random"]["mean_depth"].mean()

plt.figure(figsize=(13, 6))
plt.bar(routing_plot["ordering_id"], routing_plot["mean_depth"],
        color=bar_colors, edgecolor="black", linewidth=0.5)
plt.axhline(random_avg_depth, color="#E45756", linestyle="--", linewidth=2)
plt.xlabel("Ordering ID")
plt.ylabel("Mean Routing-Stage Depth")
plt.title("Routing-Stage Circuit Depth Across Gate Orderings", fontweight="bold")
plt.xticks(range(22))
plt.grid(axis="y", alpha=0.25)
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

summary.sort_values("mean_swap").reset_index(drop=True)


## Native-basis experiment

The same set of orderings is compiled to the native basis `rz`, `sx`, `x`, `cx` to compare final circuit depth and two-qubit gate count.


In [ ]:
basis_gates = ["rz", "sx", "x", "cx"]
native_results = []

for seed in [0, 1, 2, 3, 4]:
    for i in range(22):
        circuit = QuantumCircuit(6)
        for q1, q2 in orderings[i]:
            circuit.cz(q1, q2)

        physical_native = transpile(
            circuit,
            coupling_map=coupling_map,
            optimization_level=1,
            initial_layout=[0, 1, 2, 3, 4, 5],
            routing_method="sabre",
            seed_transpiler=seed,
            basis_gates=basis_gates
        )

        ordering_type = "original" if i == 0 else ("heuristic" if i == 21 else "random")
        native_results.append({
            "ordering_id": i,
            "ordering_type": ordering_type,
            "seed": seed,
            "native_depth": physical_native.depth(),
            "cx_count": physical_native.count_ops().get("cx", 0)
        })

native_df = pd.DataFrame(native_results)
native_summary = (
    native_df.groupby(["ordering_id", "ordering_type"])
      .agg(
          mean_native_depth=("native_depth", "mean"),
          mean_cx=("cx_count", "mean"),
          std_native_depth=("native_depth", "std"),
          std_cx=("cx_count", "std")
      )
      .reset_index()
)

native_summary.sort_values("mean_cx").reset_index(drop=True)


In [ ]:
native_plot = native_summary.sort_values("ordering_id")
bar_colors = [colors[t] for t in native_plot["ordering_type"]]

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(native_plot["ordering_id"], native_plot["mean_native_depth"],
       color=bar_colors, edgecolor="black", linewidth=0.5)
ax.axhline(native_plot[native_plot["ordering_type"] == "random"]["mean_native_depth"].mean(),
           color="#E45756", linestyle="--", linewidth=2)
ax.set_xlabel("Ordering ID")
ax.set_ylabel("Mean Native Circuit Depth")
ax.set_title("Final Native Circuit Depth Across Gate Orderings", fontweight="bold")
ax.set_xticks(range(22))
ax.grid(axis="y", alpha=0.25)
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(native_plot["ordering_id"], native_plot["mean_cx"],
       color=bar_colors, edgecolor="black", linewidth=0.5)
ax.axhline(native_plot[native_plot["ordering_type"] == "random"]["mean_cx"].mean(),
           color="#E45756", linestyle="--", linewidth=2)
ax.set_xlabel("Ordering ID")
ax.set_ylabel("Mean CX Count")
ax.set_title("Final Two-Qubit Gate Count Across Gate Orderings", fontweight="bold")
ax.set_xticks(range(22))
ax.grid(axis="y", alpha=0.25)
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()


## Interpretation and limitations

This is a deliberately small exploratory experiment. Under this controlled setup, logically equivalent gate orderings can lead SABRE to different routing decisions and different final physical costs.

The experiment uses one 6-qubit synthetic circuit, one line topology, one routing method, and five transpiler seeds. It should therefore be treated as evidence of a compiler-sensitivity phenomenon worth investigating, not as a general benchmark result.
